# Grade 11 · Unit 7 — Geospatial thinking

> Points here are **SYNTHETIC** (random) inside Nepal's rough bounding box. Real maps: OpenStreetMap data are ODbL — credit '© OpenStreetMap contributors'.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points given in degrees."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

ktm, pokhara = (27.7172, 85.3240), (28.2096, 83.9856)
d = float(haversine_km(*ktm, *pokhara))
assert 130 < d < 160          # straight-line distance is roughly 140 km
print(f"Kathmandu–Pokhara straight line ≈ {d:.0f} km (road distance is longer)")

## Synthetic monitoring stations

In [ ]:
rng = np.random.default_rng(9)
n = 40
st = pd.DataFrame({"lat": rng.uniform(26.4, 30.4, n), "lon": rng.uniform(80.1, 88.1, n)})
st["pm25"] = (30 + 12 * (st["lon"] - 80) + rng.normal(0, 8, n)).round(1)       # invented pattern
plt.scatter(st["lon"], st["lat"], c=st["pm25"], cmap="viridis"); plt.colorbar(label="PM2.5 (SYNTHETIC)")
plt.xlabel("longitude"); plt.ylabel("latitude"); plt.title("Invented stations in Nepal's bounding box"); plt.show()

## Nearest station to Kathmandu

In [ ]:
st["km_to_ktm"] = haversine_km(ktm[0], ktm[1], st["lat"], st["lon"])
nearest = st.sort_values("km_to_ktm").iloc[0]
assert nearest["km_to_ktm"] >= 0
nearest

**Make:** plan (in words) a choropleth of a *rate* by district: which table, which boundary file, which licence credits?